# Pen Digits Classification

10,992 handwritten digits, each written with a pen on a tablet and described by 16 spatial coordinates of the pen trajectory. Predict the digit (0-9) from the trajectory alone — a harder cousin of the pixel-based MNIST projects.

In [1]:
import io, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def download(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r:
        return r.read()
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

# UCI Pen-Based Recognition of Handwritten Digits (id 81)
df = pd.read_csv(io.BytesIO(download('https://archive.ics.uci.edu/static/public/81/data.csv')))
print(df.shape)
print('Samples per digit:')
print(df['Class'].value_counts().sort_index())
df.head(3)


(10992, 17)
Samples per digit:
Class
0    1143
1    1143
2    1144
3    1055
4    1144
5    1055
6    1056
7    1142
8    1055
9    1055
Name: count, dtype: int64


,Attribute1,Attribute2,Attribute3,Attribute4,Attribute5,Attribute6,Attribute7,Attribute8,Attribute9,Attribute10,Attribute11,Attribute12,Attribute13,Attribute14,Attribute15,Attribute16,Class
0,47,100,27,81,57,37,26,0,0,23,56,53,100,90,40,98,8
1,0,89,27,100,42,75,29,45,15,15,37,0,69,2,100,6,2
2,0,57,31,68,72,90,100,100,76,75,50,51,28,25,16,0,1


In [2]:

feats = [c for c in df.columns if c != 'Class']
X = df[feats]; y = df['Class']
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1).fit(Xtr, ytr)
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
acc_rf = accuracy_score(yte, rf.predict(Xte)); acc_lr = accuracy_score(yte, lr.predict(Xte))
print(f'Random Forest accuracy: {acc_rf:.4f}')
print(f'Logistic Regression accuracy: {acc_lr:.4f}')
print(classification_report(yte, rf.predict(Xte)))


Random Forest accuracy: 0.9905
Logistic Regression accuracy: 0.9454
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       285
           1       0.99      0.96      0.98       286
           2       0.96      0.99      0.98       286
           3       0.99      0.99      0.99       264
           4       0.99      1.00      0.99       286
           5       1.00      1.00      1.00       264
           6       1.00      1.00      1.00       264
           7       0.98      0.99      0.99       285
           8       1.00      0.99      1.00       264
           9       0.99      0.99      0.99       264

    accuracy                           0.99      2748
   macro avg       0.99      0.99      0.99      2748
weighted avg       0.99      0.99      0.99      2748



In [3]:
imp = pd.Series(rf.feature_importances_, index=feats).sort_values(ascending=False)
print(imp.head(20))
imp.head(20).plot(kind='barh', title='Which trajectory coordinates matter most?')
plt.tight_layout()
plt.show()
winner = rf if acc_rf >= acc_lr else lr
winner_name = 'Random Forest' if acc_rf >= acc_lr else 'Logistic Regression'
print(f'Winner: {winner_name}')
cm = confusion_matrix(yte, winner.predict(Xte), labels=sorted(y.unique()))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=sorted(y.unique()))
disp.plot(xticks_rotation=45)
plt.title(f'Confusion matrix - {winner_name}')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()


Attribute14    0.108120
Attribute16    0.101584
Attribute15    0.079000
Attribute11    0.074106
Attribute10    0.073335
Attribute5     0.071493
Attribute9     0.066406
Attribute8     0.061427
Attribute1     0.059850
Attribute2     0.052616
Attribute4     0.050811
Attribute6     0.050577
Attribute12    0.049606
Attribute7     0.043604
Attribute13    0.030239
Attribute3     0.027227
dtype: float64
Winner: Random Forest
